In [1]:
import requests
import time
import os
import pandas as pd
import re
import unicodedata
from difflib import SequenceMatcher

from numpy import rec
from pandas.core.methods import describe

## Konfiguracija i priprema za prikupljanje podataka

U ovom segmentu koda definisala sam osnovne parametre za rad sa API-jem, listu ciljnih izvođača, kao i pametni mehanizam koji sprečava dupliranje podataka prilikom preuzimanja.

### Ključne komponente i logika:

* **Autentifikacija i API konekcija:** Definisala sam pristupne parametre (`BASE_URL` i `HEADERS`) koristeći jedinstveni `RAPIDAPI_KEY` kako bih uspešno autorizovala zahteve ka *Spotify Extended Audio Features* API-ju.
* **Kontrola protoka (Rate Limiting):** Postavila sam parametre `DELAY = 0.4` (pauza od 0.4 sekunde između poziva) i `BATCH_SIZE = 5`. Ovo je bitno kako bih sprečila preopterećenje servera i izbegla blokiranje od strane API-ja.
* **Defisanje skupa podataka (`SVI_UMETNICI`):** Kreirala sam statičku listu koja sadrži jedinstvene Spotify ID-eve i nazive za 22 izvođača koje sam ručno prikupila.
* **Mehanizam za nastavak rada :**
  * Skripta prvo proverava da li već postoji generisani fajl `spotify_audio_features.csv`.
  * Ukoliko fajl postoji, učitavam ga pomoću biblioteke `pandas` i izdvajam jedinstvene nazive umetnika koji su već uspešno procesuirani (`vec_prikupljeni`).
  * Na osnovu toga, filtriram početnu listu i kreiram listu `preostali`. Ovaj korak omogućava skripti da, u slučaju prekida interneta ili gašenja programa, nastavi rad tačno tamo gde je stala, čime štedim vreme i API resurse.

In [2]:

RAPIDAPI_KEY = "KLJUC"

BASE_URL = "https://spotify-extended-audio-features-api.p.rapidapi.com/v1"
HEADERS  = {
    "x-rapidapi-key":  RAPIDAPI_KEY,
    "x-rapidapi-host": "spotify-extended-audio-features-api.p.rapidapi.com"
}

MARKET     = "GB"   # Tržište Velike Britanije
BATCH_SIZE = 5      # RapidAPI nam dozvoljava samo 5 odjednom
DELAY      = 0.4    # Pauza između poziva u sekundama
OUTPUT_CSV = "spotify_audio_features.csv"

# Kompletan spisak svih 22 umetnika koji su ručno prikupljeni
SVI_UMETNICI = [
    {"id": "2DaxqgrOhkeH0fpeiQq2f4", "name": "Oasis"},
    {"id": "7MhMgCo0Bl0Kukl93PZbYS", "name": "Blur"},
    {"id": "6PHIK3kjWggLtVygsOtpqS", "name": "Suede"},
    {"id": "36E7oYfz3LLRto6l2WmDcD", "name": "Pulp"},
    {"id": "3l14gV4hIMAjmo7KUvEWTx", "name": "Elastica"},
    {"id": "0sHeX8oQ6o7xic3wMf4NBU", "name": "Supergrass"},
    {"id": "3iejrAcqxYoVgyxp6zkWgs", "name": "Shed Seven"},
    {"id": "0NbfEPYRgMczimdfM3skmH", "name": "Sleeper"},
    {"id": "414ztMLgMt8KAC6Ap5IZtN", "name": "Menswear"},
    {"id": "6bEiU0j4lC3yh2D1kHzFpb", "name": "Echobelly"},
    {"id": "2uH0RyPcX7fnCcT90HFDQX", "name": "Manic Street Preachers"},
    {"id": "2cGwlqi3k18jFpUyTrsR84", "name": "The Verve"},
    {"id": "5fScAXreYFnuqwOgBsJgSd", "name": "The Charlatans"},
    {"id": "2evydP72Z45DouM4uMGsIE", "name": "Ash"},
    {"id": "3ysp8GwsheDcBxP9q65lBg", "name": "Lush"},
    {"id": "47Z8LEl3LnQkcpva0xSthT", "name": "The La's"},
    {"id": "6sN51vEARnAAdBw1IKZ8Q9", "name": "Liam Gallagher"},
    {"id": "0O98jlCaPzvsoei6U5jfEL", "name": "Damon Albarn"},
    {"id": "13W7XLRXdWeLmIu9vacE1w", "name": "Jarvis Cocker"},
    {"id": "0ndVVO80abvgGgjv2ICzct", "name": "Brett Anderson"},
    {"id": "7Lf3LOZp3U3u2f6cWMd3AH", "name": "Paul Weller"},
    {"id": "3bUwxJgNakzYKkqAVgZLlh", "name": "Travis"},
]

# Automatski proveravamo ko je već u CSV-u da ne bismo duplirali podatke
vec_prikupljeni = set()
if os.path.exists(OUTPUT_CSV):
    try:
        df_postojeci = pd.read_csv(OUTPUT_CSV)
        if "artist_name" in df_postojeci.columns:
            vec_prikupljeni = set(df_postojeci["artist_name"].unique())
    except Exception:
        pass

preostali = [u for u in SVI_UMETNICI if u["name"] not in vec_prikupljeni]

print(f"Konfiguracija učitana.")
print(f"   Već prikupljeni ({len(vec_prikupljeni)}): {sorted(list(vec_prikupljeni))}")
print(f"   Preostali za prikupljanje ({len(preostali)})")

Konfiguracija učitana.
   Već prikupljeni (22): ['Ash', 'Blur', 'Brett Anderson', 'Damon Albarn', 'Echobelly', 'Elastica', 'Jarvis Cocker', 'Liam Gallagher', 'Lush', 'Manic Street Preachers', 'Menswear', 'Oasis', 'Paul Weller', 'Pulp', 'Shed Seven', 'Sleeper', 'Suede', 'Supergrass', 'The Charlatans', "The La's", 'The Verve', 'Travis']
   Preostali za prikupljanje (0)


## Implementacija funkcija za ekstrakciju i obradu podataka

U ovom koraku sam pripremila funkcije koje će mi biti potrebne za izvlačenje podataka.

### Glavne funkcionalnosti koje sam implementirala:

1. **Upravljanje API limitima (Rate Limiting, Retries):** Centralizovala sam slanje HTTP zahteva kako bih automatski prepoznala i rešila grešku `429` (Previše zahteva) privremenim zaustavljanjem skripte na 15 sekundi.
2. **Preuzimanje podataka u delovima:** Omogućila sam skripti da automatski pomera marker (`offset`) i preuzima albume u paketima, sve dok ne povuče kompletnu diskografiju umetnika.
3. **Grupna obrada:** Implementirala sam podelu velikih lista na manje pakete, što optimizuje broj API poziva jer endpoint za audio karakteristike dozvoljava slanje više ID-jeva odjednom.
4. **Čuvanje podataka i sprečavanje duplikata:** Kreirala sam funkciju za upis u CSV fajl, koja pre svakog upisa proverava da li podaci za tog umetnika već postoje, čime se eliminiše rizik od dupliranja podataka.

In [3]:
def chunks(lst, n):
    # Deli listu na grupe od n elemenata.
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


def api_get(endpoint, params=None):
    # Šalje GET zahtev ka API-ju sa automatskim hendlovanjem rate limita (429).
    url = f"{BASE_URL}{endpoint}"
    try:
        response = requests.get(url, headers=HEADERS, params=params, timeout=15)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:
            print("Rate limit pogođen - čekam 15 sekundi...") # Nakon 15 sekundi pokušava još jednom
            time.sleep(15)
            response = requests.get(url, headers=HEADERS, params=params, timeout=15)
            return response.json() if response.status_code == 200 else None
        else:
            print(f"Greška {response.status_code} za: {endpoint}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Mrežna greška: {e}")
        return None


def get_albums(umetnik):
    # Prikuplja sve studijske albume umetnika (include_groups = album)
    print(f"Prikupljam albume za: {umetnik['name']}")
    albumi, offset, limit = [], 0, 50 # Uzima po 50 albuma jer nam toliko API dozvoljava
    while True:
        data = api_get(
            f"/artists/{umetnik['id']}/albums",
            params={"include_groups": "album", "limit": limit, "offset": offset, "market": MARKET}
        )
        if not data or "items" not in data: break
        batch = data["items"]
        if not batch: break

        for album in batch:
            izvodjaci = ", ".join([a["name"] for a in album.get("artists", [])])
            albumi.append({
                "artist_id":    umetnik["id"],
                "artist_name":  umetnik["name"],
                "album_id":     album["id"],
                "album_name":   album["name"],
                "album_type":   album.get("album_type"),
                "release_date": album.get("release_date"),
                "total_tracks": album.get("total_tracks"),
                "artists":      izvodjaci,
            })
        if len(batch) < limit: break
        offset += limit
        time.sleep(DELAY)
    print(f"  Pronađeno albuma: {len(albumi)}")
    return albumi


def get_tracks(albumi):
    # Dohvata osnovne podatke o svim numerama iz albuma (koristi limit 50).
    sve_numere = []
    for album in albumi:
        data = api_get(f"/albums/{album['album_id']}/tracks", params={"limit": 50, "market": MARKET})
        if not data or "items" not in data:
            time.sleep(DELAY)
            continue
        track_ids = [t["id"] for t in data["items"] if t.get("id")]
        if not track_ids:
            time.sleep(DELAY)
            continue

        # /tracks endpoint podržava do 50 id-jeva odjednom
        tracks_data = api_get("/tracks", params={"ids": ",".join(track_ids), "market": MARKET})
        if not tracks_data or "tracks" not in tracks_data:
            time.sleep(DELAY)
            continue

        for track in tracks_data["tracks"]:
            if not track: continue
            sve_numere.append({
                "track_id":     track["id"],
                "track_name":   track["name"],
                "duration_ms":  track.get("duration_ms"),
                "popularity":   track.get("popularity"),
                "explicit":     track.get("explicit", False),
                "track_number": track.get("track_number"),
                "disc_number":  track.get("disc_number", 1),
                "artist_id":    album["artist_id"],
                "artist_name":  album["artist_name"],
                "album_id":     album["album_id"],
                "album_name":   album["album_name"],
                "release_date": album["release_date"],
                "album_type":   album["album_type"],
            })
        time.sleep(DELAY)
    print(f"  Ukupno pronađeno numera: {len(sve_numere)}")
    return sve_numere


def get_audio_features(numere):
    # Prikupljamo audio features
    track_ids    = [n["track_id"] for n in numere]
    svi_features = []
    for grupa in chunks(track_ids, BATCH_SIZE):
        data = api_get("/audio-features", params={"ids": ",".join(grupa)})
        if not data or "audio_features" not in data:
            time.sleep(DELAY)
            continue
        for features in data["audio_features"]:
            if not features: continue
            svi_features.append({
                "track_id":         features.get("id"),
                "danceability":     features.get("danceability"),
                "energy":           features.get("energy"),
                "key":              features.get("key"),
                "loudness":         features.get("loudness"),
                "mode":             features.get("mode"),
                "speechiness":      features.get("speechiness"),
                "acousticness":     features.get("acousticness"),
                "instrumentalness": features.get("instrumentalness"),
                "liveness":         features.get("liveness"),
                "valence":          features.get("valence"),
                "tempo":            features.get("tempo"),
                "time_signature":   features.get("time_signature"),
                "duration_ms":      features.get("duration_ms"),
            })
        time.sleep(DELAY)
    print(f"  Uspješno skinuti features za {len(svi_features)}/{len(track_ids)} numera")
    return svi_features


def export_csv(df_final, umetnik):
    # Dodajemo podatke u zajednički CSV fajl
    file_exists = os.path.exists(OUTPUT_CSV)
    if file_exists:
        try:
            df_provera = pd.read_csv(OUTPUT_CSV)
            if umetnik["name"] in df_provera["artist_name"].values:
                print(f"{umetnik['name']} već postoji u fajlu.")
                return
        except Exception:
            pass

    df_final.to_csv(OUTPUT_CSV, mode="a", header=not file_exists, index=False, encoding="utf-8")
    print(f"Podaci za [{umetnik['name']}] uspešno dodati u {OUTPUT_CSV}")

print("Sve funkcije definisane i spremne.")

Sve funkcije definisane i spremne.


## Glavno pokretanje i spajanje svih podataka

Ovaj deo koda predstavlja glavni deo koji povezuje sve funkcije koje sam prethodno napisala. Njen glavni zadatak je da ide redom kroz preostale umetnike, preuzme sve njihove podatke (albume, pesme i audio features), sredi ih, spoji u jednu čistu tabelu i na kraju sve trajno sačuva u CSV fajl.

### Koraci:

* **Početne provere:** Pre nego što petlja uopšte krene sa radom, skripta proverava da li sam zaboravila da upišem svoj API ključ i da li je lista preostalih umetnika prazna. Ako je sve već ranije skinuto, program me odmah obavesti o tome kako ne bismo trošili API zahteve.
* **Skidanje podataka:** Unutar petlje, za svakog umetnika redom pokrećem funkcije za prikupljanje albuma, zatim pesama iz tih albuma, i na kraju njihovih detaljnih muzičkih karakteristika (audio features). Ako neka od funkcija vrati prazan rezultat, skripta pomoću komande `continue` bezbedno preskače tog izvođača i prelazi na sledećeg.
* **Čišćenje i spajanje tabela:** Kada dobijem sve podatke, pretvaram ih u Pandas tabele. Pošto i podaci o pesmama i muzičke karakteristike sadrže informaciju o trajanju pesme (`duration_ms`), namerno je brišem iz prve tabele pomoću naredbe `.drop()`. Na taj način sprečavam da nakon spajanja dobijem dve duplirane kolone u finalnom fajlu. Nakon toga, spajam ove dve tabele preko jedinstvenog ID-ja pesme (`track_id`), i to tako da u finalnoj tabeli ostanu samo pesme koje imaju kompletne podatke sa oba mesta.

In [4]:
if RAPIDAPI_KEY == "KLJUC":
    print("Greška: Popuni RAPIDAPI_KEY u pre pokretanja!")
elif not preostali:
    print("Svi umetnici sa spiska su već uspešno prikupljeni u CSV!")
else:
    print(f"Započinjem automatsko prikupljanje za preostalih {len(preostali)} umetnika...\n")

    for trenutni_umetnik in preostali:
        print("-" * 50)
        print(f"IZVOĐAČ: {trenutni_umetnik['name'].upper()}")
        print("-" * 50)

        try:
            # Prikupljanje albuma
            albumi = get_albums(trenutni_umetnik)
            if not albumi: continue

            # Prikupljanje pesama
            numere = get_tracks(albumi)
            if not numere: continue

            # Prikupljanje Audio Features
            features = get_audio_features(numere)
            if not features: continue

            # Merge i čišćenje dupliranih kolona
            df_numere = pd.DataFrame(numere)
            df_features = pd.DataFrame(features)

            df_numere_bez_dur = df_numere.drop(columns=["duration_ms"], errors="ignore")
            df_final = pd.merge(df_numere_bez_dur, df_features, on="track_id", how="inner")

            # Eksport u CSV
            export_csv(df_final, trenutni_umetnik)

            # Pauza između izvođača
            time.sleep(1.5)

        except Exception as e:
            print(f"Neočekivana greška na izvođaču {trenutni_umetnik['name']}: {e}")
            print("Nastavljam sa sledećim na listi...")
            continue

    print("\nSVE JE ZAVRŠENO!")

Svi umetnici sa spiska su već uspešno prikupljeni u CSV!


In [5]:
# Provera podataka
if os.path.exists(OUTPUT_CSV):
    df_sve = pd.read_csv(OUTPUT_CSV)
    print("PREGLED TRENUTNOG STANJA DATASETA:")
    print(f"Ukupno pesama u bazi: {len(df_sve)}")
    print(f"Broj jedinstvenih umetnika: {df_sve['artist_name'].nunique()}")
    print(f"Broj jedinstvenih albuma:   {df_sve['album_name'].nunique()}")
    print(f"\nBroj numera po umetniku:")
    print(df_sve.groupby("artist_name")["track_id"].count().sort_values(ascending=False).to_string())
else:
    print("CSV fajl još uvek ne postoji na ovoj putanji.")

PREGLED TRENUTNOG STANJA DATASETA:
Ukupno pesama u bazi: 9024
Broj jedinstvenih umetnika: 22
Broj jedinstvenih albuma:   455

Broj numera po umetniku:
artist_name
Oasis                     1455
Suede                     1150
Paul Weller                910
Manic Street Preachers     792
Ash                        757
Blur                       529
The Charlatans             474
Pulp                       456
Supergrass                 386
Shed Seven                 376
Travis                     321
The La's                   307
The Verve                  235
Brett Anderson             159
Jarvis Cocker              126
Lush                       114
Damon Albarn               113
Liam Gallagher             103
Echobelly                   86
Sleeper                     66
Elastica                    60
Menswear                    49


## Problem sa dupliranim verzijama pesama

Nakon što sam pogledala pregled trenutnog stanja i statistiku po umetnicima, primetila sam jedan problem – ukupan broj pesama u bazi je znatno veći nego što ovi bendovi realno imaju studijskih pesama.

Kada moja skripta zatraži sve albume nekog izvođača, Spotify API prati to pravilo i vraća sve moguće verzije koje su ikada okačene na platformu pod kategorijom albuma. To obuhvata standardne albuma, Remastered verzije, Deluxe i Demo izdanja, Live verzije sa koncerata...

S obzirom da je ideja bila da se u datasetu nalaze samo originalne verzije pesama, dalje sam pokušala da na najefikasniji način očistim ove podatke, bez da ručno prolazim kroz dataset jer se u njemu nalazi veliki broj pesama.

## Pomoćne funkcije za normalizaciju i čišćenje naziva pesama/albuma
Da bih uspešno grupisala i eliminisala ove duplikate, kreirala sam skup ključnih reči (`VERZIJA_RECI`) i nekoliko pomoćnih funkcija koje izoluju isključivo osnovno ime pesme ili albuma.

* **`naziv_sadrzi_verziju_rec` i `sufiks_je_verzija`**: Ove funkcije proveravaju da li tekst sadrži neku od specifičnih reči iz našeg skupa (poput *live*, *remix*, *demo*), ali striktno kao **zasebnu reč**, a ne kao podstring. Na ovaj način sprečavamo lažno pozitivne rezultate (npr. reč *"Live"* detektujemo kao oznaku verzije, dok reč *"Parklive"* ostaje netaknuta).
* **`izvuci_osnovno_ime_pesme`**: Glavna funkcija za transformaciju stringova. Implementirala sam Regex (regularni izraz) koji skenira i uklanja sve unutar oblih `()` ili uglastih `[]` zagrada, ali samo ako se unutar njih nalazi neka od reči iz skupa verzija. Nakon toga, funkcija bezbedno uklanja i preostale sufikse iza crtica.

**Primeri transformacije:**
* `"Wonderwall - Remastered"` - `"Wonderwall"`
* `"Definitely Maybe (Remastered) [Deluxe Version]"` - `"Definitely Maybe"`
* `"Live Forever - Live at Knebworth"` - `"Live Forever"`

Ovim procesom obezbeđujem da u kasnijim koracima poredim isključivo čiste, normalizovane nazive.

In [6]:
VERZIJA_RECI = {
    "live", "remaster", "remastered", "demo", "acoustic", "unplugged",
    "remix", "edit", "mix", "version", "mono", "stereo", "bonus",
    "instrumental", "session", "radio", "deluxe", "anniversary",
    "expanded", "collectors", "legacy", "rarities", "unreleased",
    "special", "bbc", "reprise", "outtake", "instore", "rethink",
    "excerpt", "strings", "cover", "broadcast",
}

# POMOĆNE FUNKCIJE

def naziv_sadrzi_verziju_rec(naziv: str) -> bool:
    # Vraća True ako naziv albuma ili pesme sadrži neku od verzija reči kao ZASEBNU reč (ne kao podstring).

    reci = re.split(r"[\s,/\-\(\)\[\]]+", naziv.lower())
    return any(r in VERZIJA_RECI for r in reci if r)


def sufiks_je_verzija(sufiks: str) -> bool:
    # True ako sufiks (deo posle ' - ' ili unutar zagrada) označava verziju.
    reci = re.split(r"[\s,/\-]+", sufiks.lower())
    return any(r in VERZIJA_RECI for r in reci if r)


def izvuci_osnovno_ime_pesme(track_name: str) -> str:
    t = str(track_name).strip()

    # Ukloni sve u zagradama () ili [] ako unutar njih postoji neka verzija reči
    pattern_zagrade = r"[\(\[][^\)\]]*(?:" + "|".join(VERZIJA_RECI) + r")[^\)\]]*[\)\]]"
    t = re.sub(pattern_zagrade, "", t, flags=re.IGNORECASE)

    # Ukloni sufikse iza crtice ' - ' ako sadrže verziju reči
    if " - " in t:
        delovi = t.split(" - ")
        if sufiks_je_verzija(delovi[-1]):
            t = " - ".join(delovi[:-1])

    return t.strip()

* **`ocisti_tekst_za_poredjenje`**: Ova funkcija priprema stringove za matematičko poređenje. Kako bi se izbeglo da algoritam vidi dve iste pesme kao različite zbog interpunkcije ili specifičnih karaktera, funkcija radi sledeće:
  * Pretvara `’s` ili `'s` u obično `s`
  * Slova sa kvačicama i akcentima svodi na njihove osnovne ekvivalente
  * Prebacuje sav tekst u mala slova i uklanja sve karaktere osim alfanumerika i razmaka.
  * Na kraju, pomoću `.split()` i `" ".join()` uklanja sve višestruke razmake unutar teksta.

* **`live_je_u_sufiksu`**: Spotify datasetovi često sadrže pesme čiji nazivi počinju rečju *Live* (npr. Oasis — *"Live Forever"*). Da algoritam ne bi greškom obrisao originalnu pesmu poredeći je sa njenim stvarnim koncertnim izvođenjem (*"Live Forever - Live at Knebworth"*), ova funkcija vraća `True` samo ako se reč "live" nalazi u sufiksu (iza crtice ili u zagradama), čime izoluje koncertna reizdanja od studijskih originala.

* **`ima_verziju_u_sufiksu`**: Pomoćna provera koja detektuje da li pesma uopšte ima oznaku alternative ili reizdanja u svom sufiksu. Ova informacija nam direktno služi za dodeljivanje prioriteta (`track_priority`) prilikom sortiranja pre čišćenja duplikata.

* **`tekstualna_slicnost`**: Koristi `SequenceMatcher` za računanje Getman-Damerau (ili sličnog) racia sličnosti između dva stringa. Na osnovu ovog skora (npr. $> 0.85$) donosi se odluka da li su dve pesme zapravo ista numera sa minimalnom razlikom u kucanju.

* **`fix_and_convert_date`**: Standardizuje datume izlaska. Problem koji se pojavljuje u mom datasetu jeste mešanje punih datuma(`YYYY-MM-DD`) sa podacima koji sadrže samo godinu izdanja (`YYYY`). Ovde eksplicitno parsiramo samo godine i postavljamo ih na prvi januar (`01-01`), kako bismo mogli da ih uporedimo

In [7]:
def ocisti_tekst_za_poredjenje(tekst: str) -> str:
    # Normalizuje tekst za poređenje

    if pd.isna(tekst) or not tekst:
        return ""
    tekst = str(tekst)
    tekst = re.sub(r"['\u2019]s\b", "s", tekst, flags=re.IGNORECASE)
    tekst = unicodedata.normalize("NFD", tekst)
    tekst = "".join(c for c in tekst if unicodedata.category(c) != "Mn")
    tekst = tekst.lower()
    tekst = re.sub(r"[^a-z0-9\s]", "", tekst)
    return " ".join(tekst.split())


def live_je_u_sufiksu(track_name: str) -> bool:

    t = str(track_name)
    if " - " in t:
        idx = t.find(" - ")
        if re.search(r'\blive\b', t[idx + 3:].lower()):
            return True
    for match in re.finditer(r'[\(\[\{](.*?)[\)\]\}]', t):
        if re.search(r'\blive\b', match.group(1).lower()):
            return True
    return False


def ima_verziju_u_sufiksu(track_name: str) -> bool:

    t = str(track_name)
    if " - " in t:
        idx = t.find(" - ")
        if sufiks_je_verzija(t[idx + 3:]):
            return True
    for match in re.finditer(r'[\(\[\{](.*?)[\)\]\}]', t):
        if sufiks_je_verzija(match.group(1)):
            return True
    return False


def tekstualna_slicnost(s1: str, s2: str) -> float:
    return SequenceMatcher(None, s1, s2).ratio()


def fix_and_convert_date(date_str) -> pd.Timestamp:
    if pd.isna(date_str):
        return pd.NaT
    date_str = str(date_str).strip()
    if len(date_str) == 4 and date_str.isdigit():
        return pd.to_datetime(f"{date_str}-01-01")
    return pd.to_datetime(date_str, errors="coerce")

### Glavna funkcija za čišćenje

#### 1. Preprocesiranje i stabilizacija datuma
Na samom početku vrši se parsiranje kolone `release_date`. S obzirom na to da Spotify za starija izdanja često ima upisanu samo godinu, svi nepotpuni datumi se standardizuju na prvi januar te godine (`01-01`).

#### 2. Korak 1: Filtriranje na nivou albuma
Prva velika eliminacija radi se nad celim albumima kako bi se u startu smanjio prostor za pretragu duplikata:
* Algoritam grupiše sve albume jednog izvođača prema njihovom očišćenom, baznom nazivu.
* Ako unutar iste grupe detektuje i originalni album i verzije, sve verzije se automatski markiraju za brisanje.
* Osiguranje od potpunog brisanja: Ukoliko u datasetu uopšte ne postoji bazični original agregiramo podatke kako bismo izvukli najraniji datum izlaska za svaki album. Potom sortiramo dostupne verzije i zadržavamo hronološki najstarije izdanje, dok ostala reizdanja brišemo. Time osiguravamo da ne ostanemo bez kompletne diskografije, prateći istorijski tok izdanja.

#### 3. Korak 2: Čišćenje na nivou pojedinačnih pesama
Nakon što su uklonjeni očigledni duplikati albuma, prelazi se na analizu pesama unutar svakog izvođača.

Pre pokretanja dvostruke petlje, DataFrame se sortira po ključu `[artist_name, track_priority, parsed_date, popularity]`. Ključni korak ovde je postavljanje `track_priority=0` (originalne pesme bez sufiksa) na sam početak. Time osiguravamo da u petlji originalna pesma uvek dobije indeks `i` (koji se zadržava), dok će njene verzije dobiti indeks `j` i biti prepoznate kao duplikat.
Za svaku pesmu proveravamo njene parove. Ako pesme pripadaju istoj "live" kategoriji (obe su studijske ili obe uživo) i ako je njihova tekstualna sličnost nakon normalizacije veća od 80% (ili se jedna kompletno sadrži u drugoj), pesma sa novijim datumom se briše.

#### 4. Korak 3: Završno sređivanje preostalih verzija
Ovaj korak služi kao sigurnosna mreža za specifične situacije koje su eventualno prošle kroz Korak 2. Ako pesma u svom nazivu ima oznaku verzije, a u očišćenom skupu pesama tog izvođača već postoji čist original sa istim baznim imenom, ta verzija se definitivno uklanja.

Na samom kraju, funkcija izbacuje privremene kolone koje su služile za sortiranje (`parsed_date`, `release_year`, `track_priority`) i štampa rezime sa brojem pesama pre i nakon uspešnog čišćenja.

In [8]:
# GLAVNA FUNKCIJA

def ocisti_spotify_dataset(putanja_do_fajla: str) -> pd.DataFrame:
    df = pd.read_csv(putanja_do_fajla)
    df_clean = df.copy()

    # Parsiranje datuma
    df_clean["parsed_date"] = df_clean["release_date"].apply(fix_and_convert_date)
    df_clean["release_year"] = df_clean["parsed_date"].dt.year
    fallback_years = (
        df_clean["release_date"]
        .astype(str)
        .str.extract(r"(\d{4})")[0]
        .astype(float)
    )
    df_clean["release_year"] = df_clean["release_year"].fillna(fallback_years)

    izvodjaci = df_clean["artist_name"].unique()


    # KORAK 1: BRISANJE ALBUMA KOJI SU VERZIJE

    albumi_za_brisanje = set()

    for artist in izvodjaci:
        sub_df = df_clean[df_clean["artist_name"] == artist]
        jedinstveni_albumi = (
            sub_df.groupby(["album_name", "album_id"])["parsed_date"]
            .min()
            .reset_index()
            .to_dict("records")
        )


        # Grupiši albume po baznom imenu (bez version sufiksa)
        grupe: dict = {}
        for album in jedinstveni_albumi:
            baza = ocisti_tekst_za_poredjenje(izvuci_osnovno_ime_pesme(album["album_name"]))
            grupe.setdefault(baza, []).append(album)

        for baza, grupa in grupe.items():
            if len(grupa) == 1:
                continue  # samo jedna verzija, ne diramo ništa

            # Proveri postoji li original u grupi
            originali = [a for a in grupa if not naziv_sadrzi_verziju_rec(a["album_name"])]
            verzije = [a for a in grupa if naziv_sadrzi_verziju_rec(a["album_name"])]

            if originali:
                # Postoji original - brišemo sve deluxe/remaster verzije iz ove grupe
                for v in verzije:
                    albumi_za_brisanje.add(v["album_id"])
            else:
                grupa_sortirana = sorted(grupa, key=lambda x: x.get('parsed_date', pd.NaT))
                # Sve novije verzije (od druge stavke pa nadalje) dodajemo u brisanje
                for v in grupa_sortirana[1:]:
                    albumi_za_brisanje.add(v["album_id"])

    df_clean = df_clean[~df_clean["album_id"].isin(albumi_za_brisanje)]


    # KORAK 2: BRISANJE DUPLIKATA NA NIVOU PESAMA

    # track_priority: 0 = originalni naziv, 1 = ima version sufiks
    df_clean["track_priority"] = df_clean["track_name"].apply(
        lambda x: 1 if ima_verziju_u_sufiksu(x) else 0
    )


    df_clean = df_clean.sort_values(
        by=["artist_name", "track_priority", "parsed_date", "popularity"],
        ascending=[True, True, True, False],
    )

    pesme_za_brisanje = set()

    for artist in izvodjaci:
        sub_df = df_clean[df_clean["artist_name"] == artist]
        pesme = sub_df[["track_name", "track_id"]].to_dict("records")

        for i in range(len(pesme)):
            if pesme[i]["track_id"] in pesme_za_brisanje:
                continue
            for j in range(i + 1, len(pesme)):
                if pesme[j]["track_id"] in pesme_za_brisanje:
                    continue

                naziv_i = pesme[i]["track_name"]
                naziv_j = pesme[j]["track_name"]


                live_i = live_je_u_sufiksu(naziv_i)
                live_j = live_je_u_sufiksu(naziv_j)
                if live_i != live_j:
                    continue

                # Izvuci i normalizuj bazne nazive
                p1_clean = ocisti_tekst_za_poredjenje(
                    izvuci_osnovno_ime_pesme(naziv_i)
                )
                p2_clean = ocisti_tekst_za_poredjenje(
                    izvuci_osnovno_ime_pesme(naziv_j)
                )

                slicnost = tekstualna_slicnost(p1_clean, p2_clean)
                jedna_u_drugoj = p1_clean in p2_clean or p2_clean in p1_clean

                if slicnost > 0.80 or jedna_u_drugoj:
                    pesme_za_brisanje.add(pesme[j]["track_id"])

    df_clean = df_clean[~df_clean["track_id"].isin(pesme_za_brisanje)]


    # KORAK 3: ČIŠĆENJE PREOSTATIH VERZIJA

    verzije_za_brisanje = set()

    for artist in izvodjaci:
        sub_df = df_clean[df_clean["artist_name"] == artist]

        bazni_nazivi = {
            ocisti_tekst_za_poredjenje(izvuci_osnovno_ime_pesme(ime))
            for ime in sub_df["track_name"]
        }

        for _, row in sub_df.iterrows():
            naziv = str(row["track_name"])
            if not ima_verziju_u_sufiksu(naziv):
                continue

            bazni = ocisti_tekst_za_poredjenje(izvuci_osnovno_ime_pesme(naziv))
            originalni_clean = ocisti_tekst_za_poredjenje(naziv)

            # Ako bazni naziv postoji kao posebna pesma - ova je duplikat
            if bazni in bazni_nazivi and bazni != originalni_clean:
                verzije_za_brisanje.add(row["track_id"])

    df_final = df_clean[~df_clean["track_id"].isin(verzije_za_brisanje)]

    # Čišćenje privremenih kolona
    df_final = df_final.drop(
        columns=["parsed_date", "release_year", "track_priority"],
        errors="ignore"
    )

    print("\nČišćenje uspešno završeno!")
    print(f"Broj pesama pre čišćenja  : {len(df)}")
    print(f"Broj pesama nakon čišćenja: {len(df_final)}")

    return df_final

In [9]:
# POKRETANJE

fajl_putanja = "spotify_audio_features.csv"
df_sve_ocisceno = ocisti_spotify_dataset(fajl_putanja)
df_sve_ocisceno = df_sve_ocisceno.reset_index(drop=True)
df_sve_ocisceno.to_csv("spotify_clean.csv", index=False)


Čišćenje uspešno završeno!
Broj pesama pre čišćenja  : 9024
Broj pesama nakon čišćenja: 2154


In [10]:
df_sve_ocisceno.info()

<class 'pandas.DataFrame'>
RangeIndex: 2154 entries, 0 to 2153
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   track_id          2154 non-null   str    
 1   track_name        2154 non-null   str    
 2   popularity        2154 non-null   int64  
 3   explicit          2154 non-null   bool   
 4   track_number      2154 non-null   int64  
 5   disc_number       2154 non-null   int64  
 6   artist_id         2154 non-null   str    
 7   artist_name       2154 non-null   str    
 8   album_id          2154 non-null   str    
 9   album_name        2154 non-null   str    
 10  release_date      2154 non-null   str    
 11  album_type        2154 non-null   str    
 12  danceability      2154 non-null   float64
 13  energy            2154 non-null   float64
 14  key               2154 non-null   int64  
 15  loudness          2154 non-null   float64
 16  mode              2154 non-null   int64  
 17  speech

In [11]:
# Provera dataseta

# Null vrednosti
null_po_kolonama = df_sve_ocisceno.isnull().sum()
ukupno_null = null_po_kolonama.sum()

if ukupno_null == 0:
    print("Null vrednosti: Dataset je potpuno čist (0 null vrednosti).")
else:
    print(f"Pronađeno je ukupno {ukupno_null} null vrednosti!")
    for kolona, broj in null_po_kolonama[null_po_kolonama > 0].items():
        print(f"   - Kolona '{kolona}': {broj} null vrednosti")

# Provera preostalih duplikata po Track ID-ju
duplirani_id = df_sve_ocisceno["track_id"].duplicated().sum()
if duplirani_id == 0:
    print("Svi 'track_id' zapisi su 100% jedinstveni.")
else:
    print(f"Postoji {duplirani_id} deljenih/dupliranih 'track_id' oznaka!")
    print("Sledeći 'track_id' zapisi se pojavljuju više puta u datasetu:")

    svi_duplikati = df_sve_ocisceno[df_sve_ocisceno["track_id"].duplicated(keep=False)]

    for tid, grupa in svi_duplikati.groupby("track_id"):
        print(f"\n   -> ID: {tid}")
        for _, row in grupa.iterrows():
            print(f"      - {row['artist_name']} - {row['track_name']} (Album: {row['album_name']})")

# Provera kritičnih kolona (da li su neki stringovi greškom postali prazni)
prazni_nazivi = (df_sve_ocisceno["track_name"].astype(str).str.strip() == "").sum()
if prazni_nazivi == 0:
    print("Nema praznih tekstualnih polja u nazivima pesama.")
else:
    print(f"{prazni_nazivi} pesama ima potpuno prazan naziv nakon čišćenja!")


Null vrednosti: Dataset je potpuno čist (0 null vrednosti).
Postoji 6 deljenih/dupliranih 'track_id' oznaka!
Sledeći 'track_id' zapisi se pojavljuju više puta u datasetu:

   -> ID: 0AtAV5F0h5lm7DKHda1Gut
      - Damon Albarn - Shock'n'awe - What's It for? (Album: Bankbusted Nuclear Detergent Blues)
      - Paul Weller - Shock'n'awe - What's It for? (Album: Bankbusted Nuclear Detergent Blues)

   -> ID: 1rZLiz1huOrOyNNJ42RLeX
      - Damon Albarn - Soft Soap Flakes Kill (Album: Bankbusted Nuclear Detergent Blues)
      - Paul Weller - Soft Soap Flakes Kill (Album: Bankbusted Nuclear Detergent Blues)

   -> ID: 2Gm92Y3FLSGIslXHaHn9BV
      - Damon Albarn - Talkin' bout Degeneration (Album: Bankbusted Nuclear Detergent Blues)
      - Paul Weller - Talkin' bout Degeneration (Album: Bankbusted Nuclear Detergent Blues)

   -> ID: 2tA1zB604uG54Fjb9fhSHp
      - Damon Albarn - It's the Economy That's Stupid (Album: Bankbusted Nuclear Detergent Blues)
      - Paul Weller - It's the Economy Tha

In [12]:
# NAKNADNO ČIŠĆENJE zato što postoji 6 istih pesama kod dva izvođača (kolaboracije)
weller_mask = df_sve_ocisceno["artist_name"] == "Paul Weller"
albarn_mask = df_sve_ocisceno["artist_name"] == "Damon Albarn"

weller_pesme = df_sve_ocisceno[weller_mask]
albarn_pesme = df_sve_ocisceno[albarn_mask]

weller_mape = {
    ocisti_tekst_za_poredjenje(izvuci_osnovno_ime_pesme(ime)): index
    for index, ime in zip(weller_pesme.index, weller_pesme["track_name"])
}

albarn_za_izbacivanje = []

for idx_albarn, row_albarn in albarn_pesme.iterrows():
    baza_albarn = ocisti_tekst_za_poredjenje(izvuci_osnovno_ime_pesme(row_albarn["track_name"]))

    # Ako Damon Albarn ima pesmu koja postoji i kod Paul Weller-a
    if baza_albarn in weller_mape:
        idx_weller = weller_mape[baza_albarn]

        df_sve_ocisceno.loc[idx_weller, "artist_name"] = "Paul Weller, Damon Albarn"

        # SPAJANJE ID-jeva
        stari_id = df_sve_ocisceno.loc[idx_weller, "track_id"]
        novi_id = f"{stari_id}, {row_albarn['track_id']}"
        df_sve_ocisceno.loc[idx_weller, "track_id"] = novi_id

        albarn_za_izbacivanje.append(idx_albarn)

# Izbacujemo višak
df_sve_ocisceno = df_sve_ocisceno.drop(albarn_za_izbacivanje)
df_sve_ocisceno = df_sve_ocisceno.reset_index(drop=True)
print(f"Uspešno spojeno {len(albarn_za_izbacivanje)} kolaborativnih pesama!")

Uspešno spojeno 6 kolaborativnih pesama!


In [13]:
# Lista svih pesama po albumima i izvodjacima
for izvodjac, grupa_izvodjaca in df_sve_ocisceno.groupby("artist_name"):
    print(f"\n{'='*50}")
    print(f"  {izvodjac}")
    print(f"{'='*50}")

    for album, grupa_albuma in grupa_izvodjaca.groupby("album_name"):
        print(f"  [{len(grupa_albuma)} pesama]  {album}")
        for _, row in grupa_albuma.iterrows():
            print(f"      - {row['track_name']}")


  Ash
  [12 pesama]  1977
      - Girl from Mars
      - Goldfinger
      - Oh Yeah
      - Kung Fu
      - Angel Interceptor
      - Lose Control
      - Lost in You
      - I'd Give You Anything
      - Gone the Dream
      - Let It Flow
      - Darkside Lightside
      - Innocent Smile
  [16 pesama]  A-Z (Vol. 2)
      - Dare To Dream
      - Mind Control
      - Insects
      - Binary
      - Physical World
      - Spheres
      - Instinct
      - Summer Snow
      - Carnal Love
      - Embers
      - Change Your Name
      - Sky Burial
      - There Is Hope Again
      - Teenage Wildlife
      - Spellbound
      - Nightfall
  [18 pesama]  A-Z Vol. 1
      - Return of White Rabbit
      - True Love 1980
      - Joy Kicks Darkness
      - Arcadia
      - Tracers
      - The Dead Disciples
      - Pripyat
      - Ichiban
      - Space Shot
      - Neon
      - Command
      - Song Of Your Desire
      - Dionysian Urge
      - War With Me
      - Coming Around Again
      - The Creep

In [14]:
# 1. Proveri da li je ostala koja pesma sa version rečju u nazivu
preostale_verzije = df_sve_ocisceno[df_sve_ocisceno['track_name'].apply(ima_verziju_u_sufiksu)]
print(f"Preostale verzije koje bi možda trebalo ukloniti: {len(preostale_verzije)}")
print(preostale_verzije[['artist_name','track_name','release_date']])

# 2. Proveri da nema pesama sa identičnim baznim nazivom kod istog izvođača
df_sve_ocisceno['bazni_naziv'] = df_sve_ocisceno['track_name'].apply(
    lambda x: ocisti_tekst_za_poredjenje(izvuci_osnovno_ime_pesme(x))
)
duplikati_baze = df_sve_ocisceno[
    df_sve_ocisceno.duplicated(subset=['artist_name','bazni_naziv'], keep=False)
]
print(f"\nPesme sa istim baznim nazivom koje su OSTALE (mogući propust):")
print(duplikati_baze[['artist_name','track_name','release_date']].sort_values('artist_name'))

Preostale verzije koje bi možda trebalo ukloniti: 3
      artist_name                                         track_name  \
1327  Paul Weller  Thinking Of You - Live Radio Session - RBB Sen...   
1328  Paul Weller  One Way Road - Live Radio Session - RBB Sendesaal   
1329  Paul Weller  If I Could Only Be Sure - Live Radio Session -...   

     release_date  
1327   2004-01-01  
1328   2004-01-01  
1329   2004-01-01  

Pesme sa istim baznim nazivom koje su OSTALE (mogući propust):
Empty DataFrame
Columns: [artist_name, track_name, release_date]
Index: []


In [18]:
pesme_za_izbacivanje = [
    "Thinking Of You - Live Radio Session - RBB Sendesaal",
    "One Way Road - Live Radio Session - RBB Sendesaal",
    "If I Could Only Be Sure - Live Radio Session - RBB Sendesaal",
]

df_sve_ocisceno = df_sve_ocisceno[
    ~(
        (df_sve_ocisceno["artist_name"] == "Paul Weller") &
        (df_sve_ocisceno["track_name"].isin(pesme_za_izbacivanje))
    )
].reset_index(drop=True)

print(f"Preostalo pesama: {len(df_sve_ocisceno)}")

Preostalo pesama: 2145


In [19]:
# Ukloni bazni_naziv kolonu
df_sve_ocisceno = df_sve_ocisceno.drop(columns=['bazni_naziv'], errors='ignore')

# Kreiraj duration_s kolonu
df_sve_ocisceno['duration_s'] = (df_sve_ocisceno['duration_ms'] / 1000).round(2)

# Pregrupiši kolone
df_sve_ocisceno = df_sve_ocisceno[[
    'track_id', 'track_name', 'track_number', 'disc_number',
    'artist_id', 'artist_name', 'album_id', 'album_name', 'album_type',
    'release_date', 'duration_ms', 'duration_s', 'popularity', 'explicit',
    'danceability', 'energy', 'key', 'loudness', 'mode',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'time_signature'
]]

df_sve_ocisceno.to_csv("spotify_clean.csv", index=False)
print(f"Sačuvano. Kolone: {list(df_sve_ocisceno.columns)}")
print(f"Broj pesama: {len(df_sve_ocisceno)}")

Sačuvano. Kolone: ['track_id', 'track_name', 'track_number', 'disc_number', 'artist_id', 'artist_name', 'album_id', 'album_name', 'album_type', 'release_date', 'duration_ms', 'duration_s', 'popularity', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']
Broj pesama: 2145
